# **Day 37: Exception Handling (try and except)**

An **exception** is an error that happens **while the program is running**. If we do not handle it, the program **crashes** and stops.

With `try` and `except`, we can **catch** the error and do something sensible instead of crashing.

Think of it like a **safety net** under a tightrope walker.

In [1]:
# print(10 / 0)        # This will raise a ZeroDivisionError and stop the program

try:
    print(10 / 0)
except ZeroDivisionError:
    print("Cannot divide by zero")

print("The program continues")

Cannot divide by zero
The program continues


## **Common Exceptions**

In [2]:
tests = [
    ("int('abc')", lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ("{'a': 1}['b']", lambda: {"a": 1}["b"]),
    ("'2' + 2", lambda: "2" + 2),
    ("open('nope.txt')", lambda: open("nope.txt")),
    ("None.upper()", lambda: None.upper()),
]
for code, test in tests:
    try:
        test()
    except Exception as e:
        print(f"{code:<18} -> {type(e).__name__}: {e}")

int('abc')         -> ValueError: invalid literal for int() with base 10: 'abc'
[1, 2][5]          -> IndexError: list index out of range
{'a': 1}['b']      -> KeyError: 'b'
'2' + 2            -> TypeError: can only concatenate str (not "int") to str
open('nope.txt')   -> FileNotFoundError: [Errno 2] No such file or directory: 'nope.txt'
None.upper()       -> AttributeError: 'NoneType' object has no attribute 'upper'


| Exception | When it happens |
|---|---|
| `ValueError` | Right type, but wrong value (`int("abc")`) |
| `IndexError` | Position does not exist in a list |
| `KeyError` | Key does not exist in a dictionary |
| `TypeError` | Wrong type (adding text and a number) |
| `FileNotFoundError` | File does not exist |
| `AttributeError` | Method or attribute does not exist |
| `ZeroDivisionError` | Dividing by zero |

## **Handling Different Errors Differently**

In [3]:
def safe_divide(a, b):
    try:
        return int(a) / int(b)
    except ZeroDivisionError:
        return "b is zero"
    except ValueError:
        return "Not a number"

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide("ten", 2))

5.0
b is zero
Not a number


## **else and finally**
- **else:** Runs only if **no** error happened.
- **finally:** Runs **always**, error or not. It is used for cleanup work like closing files.

In [4]:
def read_number(s):
    try:
        n = int(s)
    except ValueError:
        print(f"  '{s}' is not a number")
    else:
        print(f"  Got the number {n}")
    finally:
        print("  (finally block ran)")

read_number("42")
read_number("forty")

  Got the number 42
  (finally block ran)
  'forty' is not a number
  (finally block ran)


## **Example from Remote Sensing: NDVI Without Crashing**
When calculating NDVI for many pixels, some pixels may have **Red + NIR = 0** (for example, pixels outside the image area). Dividing by zero would crash the program. We catch this error and give such pixels a **NoData** value.

In [5]:
NODATA = -9999

def safe_ndvi(nir, red):
    try:
        return round((nir - red) / (nir + red), 3)
    except ZeroDivisionError:
        return NODATA

pixels = [(0.45, 0.05), (0.0, 0.0), (0.30, 0.10), (0.02, 0.06), (0.0, 0.0)]
results = [safe_ndvi(n, r) for n, r in pixels]
print(results)

valid = [v for v in results if v != NODATA]
print("Valid pixels:", len(valid), " Average NDVI:", round(sum(valid) / len(valid), 3))

[0.8, -9999, 0.5, -0.5, -9999]
Valid pixels: 3  Average NDVI: 0.267


## **Example from GIS: Reading Messy Coordinates**
Data typed by people often has mistakes. Let us read coordinates and **skip** the bad ones instead of crashing.

In [6]:
raw_rows = ["22.57,88.36", "26.72,88.39", "abc,88.1", "22.42", "23.52,87.31", ",", "27.04,88.26"]

good, bad = [], []
for row in raw_rows:
    try:
        lat_text, lon_text = row.split(",")
        lat, lon = float(lat_text), float(lon_text)
        good.append((lat, lon))
    except ValueError as e:
        bad.append((row, str(e)))

print("Good points:", good)
print("Skipped:")
for row, reason in bad:
    print(f"  {row!r:<14} -> {reason}")

Good points: [(22.57, 88.36), (26.72, 88.39), (23.52, 87.31), (27.04, 88.26)]
Skipped:
  'abc,88.1'     -> could not convert string to float: 'abc'
  '22.42'        -> not enough values to unpack (expected 2, got 1)
  ','            -> could not convert string to float: ''


- `"22.42"` has only one value, so unpacking into two variables gives a `ValueError`.
- `"abc"` cannot be converted to float, which is also a `ValueError`.
- `e` holds the error message, which helps us understand what went wrong.

## **Input Validation Loop**
A common pattern: keep asking until the user types a valid value.

In [7]:
# while True:
#     try:
#         age = int(input("Age: "))
#         break
#     except ValueError:
#         print("Please type a number")

inputs = iter(["abc", "twenty", "21"])     # Pretend the user typed these
while True:
    raw = next(inputs)
    try:
        age = int(raw)
        break
    except ValueError:
        print(raw, "-> Please type a number")
print("Age:", age)

abc -> Please type a number
twenty -> Please type a number
Age: 21


## **Avoid a Bare except**
Writing only `except:` catches **every** error, even spelling mistakes in our own code. This hides real bugs. Always write the error type you expect.

In [8]:
# Quick question
# What will be printed?
# try:
#     x = [1, 2, 3][3]
# except IndexError:
#     print("A")
# else:
#     print("B")
# finally:
#     print("C")